# 🌐 Network Slicing 6G — Modélisation & Évaluation
**Phase 2 : Modeling & Evaluation**

---

## 📌 Contexte du projet

Dans les réseaux **5G/6G**, le *network slicing* crée des tranches virtuelles adaptées à chaque cas d'usage :
- **eMBB** : haut débit, streaming vidéo
- **URLLC** : chirurgie à distance, véhicules autonomes
- **mMTC** : IoT, capteurs

Chaque slice doit respecter des **SLA** stricts : latence, perte de paquets, gigue, débit.

**Problématique :** Comment anticiper les dégradations de service et garantir le respect des SLA ?

---

## 🎯 Objectifs Data Science

| # | Objectif | Type ML | Target |
|---|----------|---------|--------|
| **5.1** | Prédiction de la congestion réseau | Classification (3 classes) | `congestion_class` |
| **5.2** | Estimation de la probabilité QoS | Régression probabiliste | `QoS_Probability` ∈ [0,1] |
| **5.3** | Détection d'anomalies trafic best-effort | Détection non supervisée | Score d'anomalie |


# ⚙️ Imports & Configuration

In [ ]:
import pandas as pd
print("pandas ok")
import numpy as np
print("numpy ok")
import matplotlib.pyplot as plt
print("matplotlib ok")
import seaborn as sns
print("seaborn ok")
from xgboost import XGBClassifier
print("xgboost ok")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             f1_score, precision_score, recall_score,
                             mean_squared_error, r2_score, mean_absolute_error)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor, IsolationForest
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier, XGBRegressor
import joblib

plt.style.use('seaborn-v0_8-darkgrid')
print('✅ Bibliothèques importées avec succès')


# 📥 Chargement du Dataset

Un seul fichier est utilisé pour les 3 objectifs : `network_slicing_congestion_final.csv`

In [ ]:
df = pd.read_csv('network_slicing_congestion_final.csv', encoding='utf-8')
print(f'✅ Dataset chargé — Shape: {df.shape}')
print(f'Colonnes : {df.columns.tolist()}')


---
# 5.1 — Prédiction de la Congestion Réseau (Classification)

## Problématique

Les métriques directes de saturation (CPU, bande passante) **ne sont pas disponibles** dans le dataset.
On utilise des **indicateurs indirects** : latence, débit, gigue — révélateurs d'une contention sous-jacente.

**Objectif :** Classer l'état futur du réseau pour déclencher des actions proactives avant toute dégradation SLA.

### Classes de congestion
| Valeur | Label | Description | Action |
|--------|-------|-------------|--------|
| 0 | **Normal** | SLA respectés, réseau stable | Aucune |
| 1 | **Light** | Légère dégradation, SLA à risque | Surveillance renforcée |
| 3 | **Critical** | Violations SLA, congestion avérée | Réorientation immédiate |


## 🔍 Feature Engineering — Suppression du Data Leakage

**Problème détecté :** Une première version produisait une accuracy de 99.9% même pour la Logistic Regression → signe de **data leakage**.
Les colonnes `Aggregated_QoS_Score`, `SLA_Respected`, `Latency_Score` avaient été construites à partir de la même logique que la target.

**Solution :** On conserve uniquement les **12 features brutes** du réseau.


In [ ]:
# Colonnes à exclure (dérivées de la target → data leakage)
colonnes_a_exclure = [
    'congestion_class',
    'Latency_Gap', 'Packet_Loss_Gap', 'Jitter_Gap', 'Rate_Gap',
    'Latency_Score', 'Packet_Loss_Score', 'Jitter_Score', 'Rate_Score',
    'QoS_Probability', 'Efficiency_Index', 'Aggregated_QoS_Score',
    'SLA_Respected', 'Latency_Stress_Ratio', 'Mobility_Jitter_Impact',
    'Bandwidth_Usage_Ratio',
]
colonnes_a_exclure = [c for c in colonnes_a_exclure if c in df.columns]

X_clean = df.drop(columns=colonnes_a_exclure)
y_clean = df['congestion_class']

print('✅ Features brutes conservées :')
print(X_clean.columns.tolist())
print(f'\nShape X: {X_clean.shape}')
print(f'\nDistribution de la target:')
print(y_clean.value_counts().sort_index())


## 🔧 Prétraitement — Split, Scaling, Encoding

In [ ]:
# Filtrer la classe 2 (1 seul échantillon — non représentatif)
valid_indices = y_clean != 2
X_clean_filtered = X_clean[valid_indices]
y_clean_filtered = y_clean[valid_indices]

print(f'Après filtrage classe 2 : {X_clean_filtered.shape}')
print(f'Distribution :\n{y_clean_filtered.value_counts().sort_index()}')

# Train / test split (stratifié)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clean_filtered, y_clean_filtered,
    test_size=0.2, random_state=42, stratify=y_clean_filtered
)
print(f'\nTrain: {X_train_c.shape} | Test: {X_test_c.shape}')

# Scaling
scaler_clf = StandardScaler()
X_train_scaled = scaler_clf.fit_transform(X_train_c)
X_test_scaled  = scaler_clf.transform(X_test_c)

# Encoding des labels (0→Normal, 1→Light, 3→Critical)
class_mapping = {0: 'Normal', 1: 'Light', 3: 'Critical'}
y_train_classes = y_train_c.map(class_mapping)
y_test_classes  = y_test_c.map(class_mapping)

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train_classes)
y_test_encoded  = le.transform(y_test_classes)

print(f'\nClasses encodées : {dict(zip(le.classes_, le.transform(le.classes_)))}')
print(f'Distribution train : {pd.Series(y_train_classes).value_counts().to_dict()}')


## 🤖 Entraînement des Modèles — Objectif 5.1

On compare 4 modèles adaptés aux données tabulaires structurées.

| Modèle | Rôle |
|--------|------|
| **Logistic Regression** | Baseline linéaire |
| **Random Forest** | Ensemble de référence |
| **XGBoost** | Boosting état de l'art |
| **Gradient Boosting** | Alternative au boosting |


In [ ]:
models_clf = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost':             XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss', verbosity=0),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results_clf = {}

for name, model in models_clf.items():
    print(f'🚀 Entraînement de {name}...')
    start = time.time()
    model.fit(X_train_scaled, y_train_encoded)
    y_pred_enc = model.predict(X_test_scaled)
    y_pred     = le.inverse_transform(y_pred_enc)
    elapsed    = time.time() - start

    results_clf[name] = {
        'Accuracy':  accuracy_score(y_test_classes, y_pred),
        'F1-Score':  f1_score(y_test_classes, y_pred, average='weighted'),
        'Precision': precision_score(y_test_classes, y_pred, average='weighted'),
        'Recall':    recall_score(y_test_classes, y_pred, average='weighted'),
        'Time (s)':  round(elapsed, 4),
        'y_pred':    y_pred,
        'model':     model,
    }
    print(f'  ✅ Accuracy={results_clf[name]["Accuracy"]:.4f} | F1={results_clf[name]["F1-Score"]:.4f} | {elapsed:.2f}s\n')


## 📊 Comparaison des Modèles

In [ ]:
results_df_clf = pd.DataFrame({
    n: {k: v for k, v in r.items() if k not in ('y_pred', 'model')}
    for n, r in results_clf.items()
}).T.sort_values(by='F1-Score', ascending=False)

print('📊 Tableau comparatif — Classification Congestion')
print('='*65)
print(results_df_clf.round(4).to_string())

best_clf_name = results_df_clf.index[0]
print(f'\n🏆 Meilleur modèle : {best_clf_name}')
print(f'   F1-Score = {results_clf[best_clf_name]["F1-Score"]:.4f}')


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

results_df_clf[['Accuracy','F1-Score','Precision','Recall']].plot(kind='bar', ax=ax1, alpha=0.85)
ax1.set_title('Métriques par modèle'); ax1.set_ylabel('Score'); ax1.set_ylim(0.7, 1.05)
ax1.set_xticklabels(results_df_clf.index, rotation=25, ha='right')
ax1.legend(loc='lower right'); ax1.grid(axis='y', linestyle='--', alpha=0.5)

results_df_clf['Time (s)'].plot(kind='bar', ax=ax2, color='steelblue', alpha=0.85)
ax2.set_title("Temps d'entraînement (s)"); ax2.set_ylabel('Secondes')
ax2.set_xticklabels(results_df_clf.index, rotation=25, ha='right')
ax2.grid(axis='y', linestyle='--', alpha=0.5)

plt.suptitle('Objectif 5.1 — Classification Congestion Réseau', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
print(f'📋 Rapport de classification — {best_clf_name}')
print('='*55)
print(classification_report(y_test_classes, results_clf[best_clf_name]['y_pred']))

plt.figure(figsize=(7, 5))
cm = confusion_matrix(y_test_classes, results_clf[best_clf_name]['y_pred'], labels=le.classes_)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'Matrice de confusion — {best_clf_name}')
plt.xlabel('Prédictions'); plt.ylabel('Réalité')
plt.tight_layout(); plt.show()


## 🔄 Validation Croisée — Objectif 5.1

**StratifiedKFold (5 folds)** — préserve la distribution des classes à chaque fold.
Un écart-type faible confirme que le modèle généralise bien.


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('='*60)
print('VALIDATION CROISÉE 5-FOLD — Classification Congestion')
print('='*60)

cv_results_clf = {}
for name, model in models_clf.items():
    scores = cross_val_score(model, X_train_scaled, y_train_encoded,
                             cv=skf, scoring='f1_weighted', n_jobs=-1)
    cv_results_clf[name] = scores
    print(f'\n{name}:')
    print(f'  F1 moyen   : {scores.mean():.4f}')
    print(f'  Écart-type : {scores.std():.4f}  ← faible = modèle stable')
    print(f'  Par fold   : {np.round(scores, 4)}')


## ⚙️ Tuning des Hyperparamètres — Objectif 5.1

**GridSearchCV** sur XGBoost, métrique : F1-Score weighted.


In [ ]:
params_xgb_clf = {
    'n_estimators':  [100, 300],
    'max_depth':     [4, 6, 8],
    'learning_rate': [0.05, 0.1],
}

print('🔍 GridSearchCV (12 combinaisons × 5 folds)...')
grid_clf = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='mlogloss', verbosity=0),
    params_xgb_clf, cv=5, scoring='f1_weighted', n_jobs=-1, verbose=1
)
grid_clf.fit(X_train_scaled, y_train_encoded)

print(f'\n✅ Meilleurs paramètres : {grid_clf.best_params_}')
print(f'   Meilleur F1 (CV)     : {grid_clf.best_score_:.4f}')


In [ ]:
best_xgb_clf = grid_clf.best_estimator_
y_pred_tuned_clf = le.inverse_transform(best_xgb_clf.predict(X_test_scaled))

f1_avant = results_clf['XGBoost']['F1-Score']
f1_apres = f1_score(y_test_classes, y_pred_tuned_clf, average='weighted')

print('📊 Comparaison avant/après tuning — XGBoost')
print('='*48)
print(f'F1 avant tuning : {f1_avant:.4f}')
print(f'F1 après tuning : {f1_apres:.4f}')
print(f'Gain            : {(f1_apres - f1_avant)*100:+.2f}%')


In [ ]:
fi = pd.Series(best_xgb_clf.feature_importances_,
               index=X_clean.columns).sort_values(ascending=False)
print('📊 Importance des features (XGBoost tuné) :')
print(fi.round(4).to_string())

plt.figure(figsize=(10, 5))
fi.plot(kind='bar', color='steelblue', alpha=0.85)
plt.title('Feature Importance — XGBoost (après tuning)')
plt.ylabel('Importance'); plt.xticks(rotation=40, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout(); plt.show()


## 💾 Sauvegarde du Modèle — Objectif 5.1

In [ ]:
os.makedirs('models', exist_ok=True)

joblib.dump(best_xgb_clf,             'models/model_6G_5_1_xgboost.joblib')
joblib.dump(scaler_clf,               'models/scaler_6G_5_1.joblib')
joblib.dump(le,                       'models/encoder_6G_5_1.joblib')
joblib.dump(X_clean.columns.tolist(), 'models/features_6G_5_1.joblib')

print('✅ Modèle 5.1 sauvegardé !')
print(f'   Fichier : models/model_6G_5_1_xgboost.joblib')
print(f'   Taille  : {os.path.getsize("models/model_6G_5_1_xgboost.joblib")/1024:.1f} KB')


---
# 5.2 — Estimation de la Probabilité de Respect de la QoS (Régression)

## Problématique

Objectif : **régression probabiliste** — quantifier la confiance qu'un slice respecte ses SLA.

**Exemple :** `QoS_Probability = 0.40` → 40% de chance que le slice respecte son SLA
→ déclenche automatiquement un fallback vers un slice plus performant.

### Construction de la target `QoS_Probability`

```
gap_score_i = sigmoid(gap_i / budget_i)
QoS_Probability = mean(gap_score_latency, gap_score_packet_loss, gap_score_jitter, gap_score_rate)
```

Un gap positif (réel < budget) → sigmoid > 0.5 → bonne performance.
Un gap négatif (violation SLA) → sigmoid < 0.5 → risque élevé.


## 🔧 Construction du Dataset de Régression

In [ ]:
epsilon = 1e-9

def safe_sigmoid(x):
    x = np.clip(x, -10, 10)
    return 1 / (1 + np.exp(-x))

# Calcul des gaps SLA
df['Latency_Gap']     = df['Latency Budget (μs)']                   - df['Slice Latency (μs)']
df['Packet_Loss_Gap'] = df['Packet Loss Budget']                    - df['Slice Packet Loss']
df['Jitter_Gap']      = df['Jitter Budget (μs)']                   - df['Slice Jitter (μs)']
df['Rate_Gap']        = df['Slice Available Transfer Rate (Gbps)']  - df['Data Rate Budget (Gbps)']

# Scores normalisés par le budget
df['Latency_Score']     = safe_sigmoid(df['Latency_Gap']     / (df['Latency Budget (μs)']  + epsilon))
df['Packet_Loss_Score'] = safe_sigmoid(df['Packet_Loss_Gap'] / (df['Packet Loss Budget']   + epsilon))
df['Jitter_Score']      = safe_sigmoid(df['Jitter_Gap']      / (df['Jitter Budget (μs)']  + epsilon))
df['Rate_Score']        = safe_sigmoid(df['Rate_Gap']         / (df['Data Rate Budget (Gbps)'] + epsilon))

# Target : probabilité globale de respect du SLA
df['QoS_Probability'] = (
    df['Latency_Score'] + df['Packet_Loss_Score'] +
    df['Jitter_Score']  + df['Rate_Score']
) / 4
df['QoS_Probability'] = df['QoS_Probability'].clip(0, 1).fillna(0)

print('✅ QoS_Probability calculée')
print(df['QoS_Probability'].describe().round(4))

plt.figure(figsize=(8, 4))
df['QoS_Probability'].hist(bins=40, color='steelblue', alpha=0.85, edgecolor='white')
plt.title('Distribution de QoS_Probability (target régression)')
plt.xlabel('QoS Probability'); plt.ylabel('Fréquence')
plt.tight_layout(); plt.show()


## 🎯 Features — Objectif 5.2 : les 4 Gaps SLA

| Feature | Définition | Interprétation |
|---------|-----------|----------------|
| `Latency_Gap` | Budget − Latence réelle | > 0 : OK, < 0 : violation |
| `Packet_Loss_Gap` | Budget − Perte réelle | > 0 : OK, < 0 : violation |
| `Jitter_Gap` | Budget − Gigue réelle | > 0 : OK, < 0 : violation |
| `Rate_Gap` | Débit réel − Budget débit | > 0 : OK, < 0 : insuffisant |


In [ ]:
X_reg = df[['Latency_Gap', 'Packet_Loss_Gap', 'Jitter_Gap', 'Rate_Gap']]
y_reg = df['QoS_Probability']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

print(f'Train : {X_train_r.shape} | Test : {X_test_r.shape}')
print(f'Target — mean: {y_train_r.mean():.4f} | std: {y_train_r.std():.4f}')


## 🤖 Entraînement des Modèles — Objectif 5.2

In [ ]:
models_reg = {
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost':      XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
}

results_reg = {}

for name, model in models_reg.items():
    print(f'🚀 Entraînement de {name}...')
    t0 = time.time(); model.fit(X_train_r, y_train_r); t_train = time.time() - t0
    t0 = time.time(); y_pred_r = model.predict(X_test_r); t_pred = time.time() - t0

    r2   = r2_score(y_test_r, y_pred_r)
    rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_r))
    mae  = mean_absolute_error(y_test_r, y_pred_r)

    results_reg[name] = {
        'R² Score': r2, 'RMSE': rmse, 'MAE': mae,
        'Train Time (s)': round(t_train, 4), 'Pred Time (s)': round(t_pred, 4),
        'y_pred': y_pred_r, 'model': model
    }
    print(f'  ✅ R²={r2:.4f} | RMSE={rmse:.4f} | MAE={mae:.4f} | Train={t_train:.2f}s\n')


In [ ]:
df_compare_reg = pd.DataFrame({
    n: {k: v for k, v in r.items() if k not in ('y_pred', 'model')}
    for n, r in results_reg.items()
}).T

print('📊 Tableau comparatif — Régression QoS')
print('='*55)
print(df_compare_reg.round(4).to_string())

best_reg_name = df_compare_reg['R² Score'].idxmax()
print(f'\n🏆 Meilleur modèle : {best_reg_name} (R²={df_compare_reg.loc[best_reg_name, "R² Score"]:.4f})')


In [ ]:
y_pred_best_reg = results_reg[best_reg_name]['y_pred']
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_test_r, y_pred_best_reg, alpha=0.3, s=10, color='steelblue')
axes[0].plot([0,1],[0,1], 'r--', linewidth=1.5, label='Prédiction parfaite')
axes[0].set_xlabel('QoS_Probability réelle'); axes[0].set_ylabel('QoS_Probability prédite')
axes[0].set_title(f'Prédit vs Réel — {best_reg_name}'); axes[0].legend()

residuals = y_test_r.values - y_pred_best_reg
axes[1].hist(residuals, bins=50, color='coral', alpha=0.85, edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Résidu (réel − prédit)'); axes[1].set_ylabel('Fréquence')
axes[1].set_title('Distribution des résidus')

plt.tight_layout(); plt.show()
print(f'Résidus — mean: {residuals.mean():.5f} | std: {residuals.std():.5f}')


## 🔄 Validation Croisée — Objectif 5.2 (KFold 5 folds)

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print('='*55)
print('VALIDATION CROISÉE 5-FOLD — Régression QoS')
print('='*55)

for name, model in models_reg.items():
    scores_r2   = cross_val_score(model, X_train_r, y_train_r, cv=kf, scoring='r2', n_jobs=-1)
    scores_rmse = cross_val_score(model, X_train_r, y_train_r,
                                  cv=kf, scoring='neg_root_mean_squared_error', n_jobs=-1)
    print(f'\n{name}:')
    print(f'  R² moyen   : {scores_r2.mean():.4f}  (±{scores_r2.std():.4f})')
    print(f'  RMSE moyen : {(-scores_rmse).mean():.4f}  (±{(-scores_rmse).std():.4f})')
    print(f'  R² par fold : {np.round(scores_r2, 4)}')


## ⚙️ Tuning des Hyperparamètres — Objectif 5.2 (GridSearchCV sur XGBoost)

In [ ]:
params_xgb_reg = {
    'n_estimators':  [100, 300],
    'max_depth':     [3, 5, 7],
    'learning_rate': [0.05, 0.1],
}

print('🔍 GridSearchCV en cours...')
grid_reg = GridSearchCV(
    XGBRegressor(random_state=42, verbosity=0),
    params_xgb_reg, cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1, verbose=1
)
grid_reg.fit(X_train_r, y_train_r)

print(f'\n✅ Meilleurs paramètres : {grid_reg.best_params_}')
print(f'   Meilleur RMSE (CV)  : {-grid_reg.best_score_:.4f}')


In [ ]:
best_xgb_reg = grid_reg.best_estimator_
y_pred_tuned_reg = best_xgb_reg.predict(X_test_r)

r2_avant   = results_reg['XGBoost']['R² Score']
rmse_avant = results_reg['XGBoost']['RMSE']
r2_apres   = r2_score(y_test_r, y_pred_tuned_reg)
rmse_apres = np.sqrt(mean_squared_error(y_test_r, y_pred_tuned_reg))

print('📊 Comparaison avant/après tuning — XGBoost Regressor')
print('='*53)
print(f'R²   avant : {r2_avant:.4f}  → après : {r2_apres:.4f}   ({(r2_apres-r2_avant)*100:+.2f}%)')
print(f'RMSE avant : {rmse_avant:.4f}  → après : {rmse_apres:.4f}   ({(rmse_apres-rmse_avant)*100:+.2f}%)')


## 💾 Sauvegarde du Modèle — Objectif 5.2

In [ ]:
os.makedirs('models', exist_ok=True)

# On sauvegarde le meilleur modèle (RandomForest ou XGBoost tuné selon les résultats)
best_reg_model = results_reg[best_reg_name]['model']

joblib.dump(best_reg_model,           'models/model_6G_5_2_xgboost.joblib')
joblib.dump(X_reg.columns.tolist(),   'models/features_6G_5_2.joblib')

print('✅ Modèle 5.2 sauvegardé !')
print(f'   Fichier : models/model_6G_5_2_xgboost.joblib')
print(f'   Taille  : {os.path.getsize("models/model_6G_5_2_xgboost.joblib")/1024:.1f} KB')


---
# 5.3 — Détection d'Anomalies pour le Trafic Best-Effort

## Problématique

Identifier des comportements anormaux dans le réseau :
trafic premium mal routé vers un slice best-effort, conditions réseau inhabituelles,
ou menaces potentielles nécessitant investigation.

**Approche :** Isolation Forest (non supervisé — pas besoin de labels)

**Avantages :**
- Rapide et efficace sur grandes données
- Pas besoin de données labellisées
- Détecte les anomalies multivariées


## 🎯 Sélection des Features — Objectif 5.3

In [ ]:
# Features pertinentes pour la détection d'anomalies réseau
features_anomaly = [
    'Slice Latency (μs)',
    'Slice Packet Loss',
    'Slice Jitter (μs)',
    'Slice Available Transfer Rate (Gbps)',
    'Latency_Stress_Ratio',
    'Bandwidth_Usage_Ratio',
    'Mobility_Jitter_Impact',
    'Required Mobility',
    'Required Connectivity'
]

# Vérifier que ces colonnes existent dans le dataset
df.columns = df.columns.str.strip()
features_valides = [col for col in features_anomaly if col in df.columns]
print(f'✅ Features disponibles ({len(features_valides)}/{len(features_anomaly)}) :')
for f in features_valides:
    print(f'   - {f}')


## 🔧 Préparation des Données — Objectif 5.3

In [ ]:
df_model_anomaly = df[features_valides].dropna().copy()
print(f'Shape dataset anomalie : {df_model_anomaly.shape}')

scaler_anomaly = StandardScaler()
X_scaled_anomaly = scaler_anomaly.fit_transform(df_model_anomaly)
print('✅ Données normalisées')


## 🤖 Entraînement — Isolation Forest

In [ ]:
isolation_forest = IsolationForest(contamination=0.05, random_state=42)
isolation_forest.fit(X_scaled_anomaly)

df_model_anomaly['anomaly'] = isolation_forest.predict(X_scaled_anomaly)
df_model_anomaly['anomaly_score'] = isolation_forest.score_samples(X_scaled_anomaly)

print('Distribution des anomalies :')
print(df_model_anomaly['anomaly'].value_counts())
print(f'\n  1 = Normal | -1 = Anomalie')
print(f'  Taux d\'anomalies : {(df_model_anomaly["anomaly"] == -1).mean()*100:.1f}%')


In [ ]:
anomalies = df_model_anomaly[df_model_anomaly['anomaly'] == -1]
normales  = df_model_anomaly[df_model_anomaly['anomaly'] == 1]

print(f"Nombre d'anomalies : {len(anomalies)}")
print(f"Nombre normaux     : {len(normales)}")

# Visualisation des scores d'anomalie
plt.figure(figsize=(10, 4))
plt.hist(df_model_anomaly[df_model_anomaly['anomaly'] == 1]['anomaly_score'],
         bins=50, alpha=0.7, color='steelblue', label='Normal', edgecolor='white')
plt.hist(df_model_anomaly[df_model_anomaly['anomaly'] == -1]['anomaly_score'],
         bins=50, alpha=0.7, color='red', label='Anomalie', edgecolor='white')
plt.xlabel('Score d\'anomalie'); plt.ylabel('Fréquence')
plt.title('Distribution des scores — Isolation Forest')
plt.legend(); plt.tight_layout(); plt.show()


## 💾 Sauvegarde du Modèle — Objectif 5.3

In [ ]:
os.makedirs('models', exist_ok=True)

joblib.dump(isolation_forest,   'models/model_anomaly_eya.joblib')
joblib.dump(scaler_anomaly,     'models/scaler_anomaly_eya.joblib')
joblib.dump(features_valides,   'models/features_anomaly_eya.joblib')

print('✅ Modèle 5.3 (Isolation Forest) sauvegardé !')
print(f'   Fichier : models/model_anomaly_eya.joblib')
print(f'   Taille  : {os.path.getsize("models/model_anomaly_eya.joblib")/1024:.1f} KB')


---
# ✅ Vérification Finale — Tous les Modèles

In [ ]:
print('🔍 VÉRIFICATION FINALE DES 3 MODÈLES')
print('='*50)

model_files = {
    '5.1 — Classification Congestion': 'models/model_6G_5_1_xgboost.joblib',
    '5.2 — Régression QoS':            'models/model_6G_5_2_xgboost.joblib',
    '5.3 — Détection Anomalies':       'models/model_anomaly_eya.joblib',
}

for label, path in model_files.items():
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024
        m = joblib.load(path)
        print(f'\n✅ {label}')
        print(f'   Fichier : {path} ({size:.1f} KB)')
        print(f'   Type    : {m.__class__.__name__}')
    else:
        print(f'\n❌ {label} — FICHIER NON TROUVÉ : {path}')

print('\n🎉 TOUS LES MODÈLES SONT PRÊTS POUR L\'APPLICATION !')


---
# 📋 Conclusion Générale

## Récapitulatif des performances

### Objectif 5.1 — Classification Congestion
| Critère | Résultat |
|---------|----------|
| Meilleur modèle | **XGBoost** |
| F1-Score (test) | **0.9775** |
| Data leakage | ✅ Détecté et corrigé |
| Feature dominante | Slice Handover (58.85%) |

### Objectif 5.2 — Régression QoS
| Critère | Résultat |
|---------|----------|
| Meilleur modèle (base) | **Random Forest** (R²=0.8451) |
| Meilleur après tuning | **XGBoost tuné** (R²≈0.8537) |
| Biais résidus | ≈ 0 (non biaisé) |

### Objectif 5.3 — Détection d'Anomalies
| Critère | Résultat |
|---------|----------|
| Algorithme | **Isolation Forest** |
| Contamination | 5% (500/10000 anomalies) |
| Approche | Non supervisée |

---

## Modèles sauvegardés pour l'application

| Fichier | Usage |
|---------|-------|
| `models/model_6G_5_1_xgboost.joblib` | Objectif 5.1 — Classification |
| `models/model_6G_5_2_xgboost.joblib` | Objectif 5.2 — Régression |
| `models/model_anomaly_eya.joblib` | Objectif 5.3 — Anomalies |

*Ces fichiers `.joblib` sont directement utilisables par le backend Spring Boot via un serveur Python (FastAPI/Flask).*
